# An overview of technical analysis in systematic trading strategies returns and a novel systematic strategy yielding positive significant returns

**Authors:** Marco Basanisi, Roberto Torresetti

**Published:** 2023-05-09

**URL:** [https://doi.org/10.55214/jcrbef.v5i1.204](https://doi.org/10.55214/jcrbef.v5i1.204)

**Abstract:** This paper contributes to the literature on systematic trading strategies, in particular technical analysis profitability. We measure the profitability and forecasting power of a trend following strategy implemented in Python on a wide perimeter (205 European stocks, 11 industries, 7 major stock exchanges) over 8 years: from 2015 to 2022. The strategy signal is based on 4 moving averages and a trailing stop loss. We also introduce a mechanism based on trailing upper and lower price bounds to avoid false signals and limit transaction costs during lateral movements. We calibrate the hyper-parameters to all stocks belonging to the same industry. The returns of the strategy applied to the constituents of the top performing industries provides a total return of 20% net of transaction costs, with an annualized Sharpe ratio of 0.54, in the out of sample time window from 2020 to 2022.

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1: Configuration

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

# Configuration
symbol = 'AAPL'
start_date = '2015-01-01'
end_date = '2022-12-31'
short_window = 40
long_window = 100
trailing_stop_loss = 0.1

## Phase 2: Data Download and Feature Engineering

In [ ]:
# Download data
data = yf.download(symbol, start=start_date, end=end_date)

# Feature engineering
data['Short_MA'] = data['Close'].rolling(window=short_window, min_periods=1).mean()
data['Long_MA'] = data['Close'].rolling(window=long_window, min_periods=1).mean()
data['Signal'] = 0


## Phase 3: Signal Generation and Portfolio Construction

In [ ]:
# Generate signals
data['Signal'][short_window:] = np.where(data['Short_MA'][short_window:] > data['Long_MA'][short_window:], 1, 0)
data['Position'] = data['Signal'].shift()

# Portfolio construction
data['Strategy_Returns'] = data['Position'] * data['Close'].pct_change()

## Phase 4: Vectorized Backtest

In [ ]:
# Backtest
initial_capital = 10000.0
positions = initial_capital * data['Position']
portfolio = positions * data['Close'].pct_change()
cumulative_returns = (1 + portfolio.cumsum() / initial_capital)

## Phase 5: Performance Metrics

In [ ]:
# Calculate performance metrics
def calculate_sharpe_ratio(returns, risk_free_rate=0.0):
    excess_returns = returns - risk_free_rate
    return np.sqrt(252) * excess_returns.mean() / excess_returns.std()

def calculate_sortino_ratio(returns, risk_free_rate=0.0):
    downside_std = returns[returns < 0].std()
    excess_returns = returns - risk_free_rate
    return np.sqrt(252) * excess_returns.mean() / downside_std

def calculate_calmar_ratio(returns):
    max_drawdown = calculate_max_drawdown(returns)
    annualized_return = np.power(returns + 1, 252/len(returns)) - 1
    return annualized_return / max_drawdown

def calculate_max_drawdown(returns):
    cumulative = np.cumprod(1 + returns)
    peak = np.maximum.accumulate(cumulative)
    drawdown = (cumulative - peak) / peak
    return drawdown.min()

sharpe_ratio = calculate_sharpe_ratio(data['Strategy_Returns'].dropna())
sortino_ratio = calculate_sortino_ratio(data['Strategy_Returns'].dropna())
calmar_ratio = calculate_calmar_ratio(data['Strategy_Returns'].dropna())
max_drawdown = calculate_max_drawdown(data['Strategy_Returns'].dropna())

print(f"Sharpe Ratio: {sharpe_ratio}")
print(f"Sortino Ratio: {sortino_ratio}")
print(f"Calmar Ratio: {calmar_ratio}")
print(f"Max Drawdown: {max_drawdown}")

# Plot equity curve
plt.figure(figsize=(12, 6))
plt.plot(cumulative_returns, label='Equity Curve')
plt.title('Equity Curve')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns')
plt.legend()
plt.show()

## Phase 6: Monitoring Stub

In [ ]:
# Monitoring stub
# This section is reserved for future implementation of monitoring tools and alerts for the strategy.
# For example, you could implement real-time data fetching and signal generation, or set up alerts for significant drawdowns.